In [6]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "pyyaml", "scikit-learn", "scipy", "pyarrow",
     "--quiet", "--break-system-packages"],
    capture_output=True, text=True
)
if result.returncode != 0:
    # fallback: try uv
    result2 = subprocess.run(
        ["uv", "pip", "install", "pyyaml", "scikit-learn", "scipy", "pyarrow",
         "--python", sys.executable, "--quiet"],
        capture_output=True, text=True
    )
    print(result2.stdout or "done via uv"); print(result2.stderr[:300] if result2.stderr else "")
else:
    print("Packages ready")

Packages ready


# 02 — Feature Engineering
Build and persist user, item, and session features ready for model training.

In [7]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import pandas as pd
from utils.config import load_config

cfg = load_config()
PROJECT_ROOT = os.path.dirname(os.getcwd())
print("Config loaded. Project root:", PROJECT_ROOT)

Config loaded. Project root: c:\Users\BIPLOB GON\Google Drive\DS & Analytics\github_projects\2026\product-recommendation-systems\product-recommendation-system


## 1. Load Raw Data

In [8]:
events        = pd.read_csv(f"{PROJECT_ROOT}/data/raw/events.csv")
category_tree = pd.read_csv(f"{PROJECT_ROOT}/data/raw/category_tree.csv")

_TARGET_PROPS = {"categoryid", "790", "available"}
_chunks = []
for _path in [
    f"{PROJECT_ROOT}/data/raw/item_properties_part1.csv",
    f"{PROJECT_ROOT}/data/raw/item_properties_part2.csv",
]:
    for _chunk in pd.read_csv(_path, chunksize=500_000, dtype=str):
        _chunks.append(_chunk[_chunk["property"].isin(_TARGET_PROPS)])
item_props = pd.concat(_chunks, ignore_index=True)

print(f"Events         : {len(events):>10,}")
print(f"Item properties: {len(item_props):>10,}")
print(f"Category tree  : {len(category_tree):>10,}")

Events         :  2,756,101
Item properties:  4,082,369
Category tree  :      1,669


## 2. User Features & Interaction Matrix

In [9]:
from features.user_features import build_user_features, build_user_item_matrix

user_features   = build_user_features(events)
interactions    = build_user_item_matrix(events)

print(f"User features shape   : {user_features.shape}")
print(f"Cold-start users      : {user_features['is_cold_start'].mean():.1%}")
print(f"Interaction matrix    : {len(interactions):,} rows")
user_features.head(3)

2026-04-04 12:10:38 | INFO     | features.user_features | Building user features from 2756101 events …
2026-04-04 12:26:29 | INFO     | features.user_features | User features built: 1407580 users, 71.2% cold-start.
2026-04-04 12:26:32 | INFO     | features.user_features | User-item matrix: 2145179 interactions across 1407580 users × 235061 items.
User features shape   : (1407580, 12)
Cold-start users      : 71.2%
Interaction matrix    : 2,145,179 rows


,visitorid,n_views,n_addtocart,n_transactions,n_unique_items,weighted_interaction_score,active_days,preferred_hour,first_seen,last_seen,conversion_rate,is_cold_start
0,0,3,0,0,3,3,1,20,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,0.0,False
1,1,1,0,0,1,1,1,17,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444,0.0,True
2,2,8,0,0,4,8,1,18,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845,0.0,False


## 3. Item Features & TF-IDF Matrix

In [10]:
from features.item_features import build_item_features, build_item_tfidf_matrix

item_features               = build_item_features(item_props, category_tree)
item_ids, tfidf_mat, vec    = build_item_tfidf_matrix(item_props, max_features=500)

print(f"Item features shape : {item_features.shape}")
print(f"TF-IDF matrix shape : {tfidf_mat.shape}")
item_features.head(3)

2026-04-04 12:26:41 | INFO     | features.item_features | Building item features from 4082369 property records …
2026-04-04 12:26:58 | INFO     | features.item_features | Item features built for 417053 unique items.
2026-04-04 12:26:58 | INFO     | features.item_features | Building TF-IDF item matrix …
2026-04-04 12:29:06 | INFO     | features.item_features | TF-IDF matrix shape: (417053, 500)
Item features shape : (417053, 9)
TF-IDF matrix shape : (417053, 500)


,itemid,n_property_updates,n_properties,first_seen,last_seen,categoryid,price,available,category_depth
0,0,3,3,2015-05-10 03:00:00,2015-05-17 03:00:00,209,91200.0,False,2.0
1,1,37,3,2015-05-10 03:00:00,2015-09-13 03:00:00,1114,6120.0,False,2.0
2,10,3,3,2015-05-10 03:00:00,2015-06-28 03:00:00,1301,15120.0,False,2.0


## 4. Session Segmentation & Sequences

In [11]:
from features.session_features import build_sessions, build_session_sequences, build_session_features

sessions_df   = build_sessions(events, session_gap_hours=1.0)
sequences     = build_session_sequences(sessions_df, max_len=20, min_len=2)
session_feats = build_session_features(sessions_df)

print(f"Total sessions        : {sessions_df['session_id'].nunique():,}")
print(f"Training sequences    : {len(sequences):,}")
print(f"Session features shape: {session_feats.shape}")
sequences.head(3)

2026-04-04 12:29:16 | INFO     | features.session_features | Segmenting events into sessions (gap=1.0h) …
2026-04-04 12:29:27 | INFO     | features.session_features | Sessions built: 1726714 total, mean length 1.60 events.
2026-04-04 12:29:27 | INFO     | features.session_features | Building session sequences (max_len=20, min_len=2) …
2026-04-04 12:29:59 | INFO     | features.session_features | Session sequences built: 386099 training samples.
Total sessions        : 1,726,714
Training sequences    : 386,099
Session features shape: (1726714, 9)


,session_id,visitorid,item_sequence,target_item,session_length,has_purchase
0,0_1,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",67045,3,False
1,1000001_2,1000001,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",230432,3,False
2,1000003_1,1000003,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",150875,2,False


## 5. Persist Processed Features

In [13]:
import os, scipy.sparse as sp, pickle

PROCESSED = f"{PROJECT_ROOT}/data/processed"
os.makedirs(PROCESSED, exist_ok=True)

# Save as CSV (pyarrow incompatible with Python 3.14 in notebook kernel)
user_features.to_csv(f"{PROCESSED}/user_features.csv", index=False)
item_features.to_csv(f"{PROCESSED}/item_features.csv", index=False)
interactions.to_csv(f"{PROCESSED}/interactions.csv", index=False)
sequences.to_csv(f"{PROCESSED}/session_sequences.csv", index=False)
session_feats.to_csv(f"{PROCESSED}/session_features.csv", index=False)
sp.save_npz(f"{PROCESSED}/tfidf_matrix.npz", tfidf_mat)
item_ids.to_csv(f"{PROCESSED}/tfidf_item_ids.csv", index=False)
with open(f"{PROCESSED}/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vec, f)

print("All processed features saved to", PROCESSED)

All processed features saved to c:\Users\BIPLOB GON\Google Drive\DS & Analytics\github_projects\2026\product-recommendation-systems\product-recommendation-system/data/processed
